# Spatial Voting Analysis with socialchoicelab

This notebook walks through a typical spatial voting analysis using the
`socialchoicelab` Python package.

## Setup (first time, or after a fresh clone)

**1. Build the C library** (from the repo root in a terminal):
```bash
cmake -S . -B build && cmake --build build
```

**2. Create a virtual environment and install:**
```bash
# From the repo root:
python -m venv .venv
# Then from the python/ directory:
cd python && ../.venv/bin/pip install -e ".[dev]"
```
Cursor may offer to create the venv automatically (it lands at the repo root) — accept it, then run the `pip install` line above.

**3. Set the library path** so Python can find `libscs_api` at runtime.
Add this to your `~/.zshrc` (or run it in the terminal before launching Cursor/Jupyter):
```bash
export SCS_LIB_PATH=/path/to/socialchoicelab/build
```

**4. Select the correct kernel** in Cursor/JupyterLab — choose the `.venv`
environment (shown as `Python (python/.venv)` or similar in the kernel picker).

After setup, run the verification cell below. You should see the API version printed.
Full setup details: `python/README.md`.

In [16]:
import numpy as np
import socialchoicelab as scl

print("C API version:", scl.scs_version())

C API version: {'major': 0, 'minor': 1, 'patch': 0}


## 1 — Voter ideal points in 2D

We model a legislature with **5 voters** in a two-dimensional issue space.

In [22]:
# Flat [x0, y0, x1, y1, ...] representation.
voters = np.array([
     20.8,  79.6,   # V1
    100.4,  69.2,   # V2
    121.2, 115.2,   # V3
    100.4,   6.4,   # V4
      7.2,  20.0,   # V5
])

# Status quo only (no additional alternatives in this scenario)
sq = np.array([100.0, 100.0])

print("Status quo:", sq)

Status quo: [100. 100.]


## 2 — Reproducible randomness with StreamManager

In [23]:
sm = scl.StreamManager(master_seed=2024, stream_names=["gen", "ties"])
# Test draw
print("Sample uniform [0,1]:", sm.uniform_real("gen", 0.0, 1.0))

Sample uniform [0,1]: 0.21542934816456774


## 3 — Majority preference between two alternatives

In [24]:
# Test a candidate point against the status quo
candidate = np.array([80.0, 80.0])
candidate_beats_sq = scl.majority_prefers_2d(candidate[0], candidate[1], sq[0], sq[1], voters)

print("Candidate (80, 80) beats SQ by majority:", candidate_beats_sq)

Candidate (80, 80) beats SQ by majority: True


## 4 — Winset: which policies beat the status quo?

In [25]:
ws = scl.winset_2d(float(sq[0]), float(sq[1]), voters)
print("Winset empty:", ws.is_empty())
print("Bounding box:", ws.bbox())

Winset empty: False
Bounding box: {'min_x': 6.799145303047595, 'min_y': -2.185084214665949, 'max_x': 125.4935131759666, 'max_y': 100.00085469695242}


In [21]:
import socialchoicelab.plots as sclp

fig = sclp.plot_spatial_voting(
    voters,
    sq          = sq,
    voter_names = [f"V{i}" for i in range(5)],
    title       = "Winset of the status quo",
)
fig = sclp.layer_winset(fig, ws)
fig.show()

NameError: name 'alts' is not defined

## 5 — Copeland winner (finite alternative set)

In [ ]:
alts = sq  # single alternative: the status quo
scores = scl.copeland_scores_2d(alts, voters)
winner = scl.copeland_winner_2d(alts, voters)
print("Copeland scores:", scores)
print("Copeland winner (0-based):", winner)

Copeland scores: [ 2 -2  0]
Copeland winner (0-based): 0


## 6 — Condorcet winner

In [ ]:
alts = sq  # single alternative: the status quo
has_cw = scl.has_condorcet_winner_2d(alts, voters)
cw     = scl.condorcet_winner_2d(alts, voters)
print("Has Condorcet winner:", has_cw)
print("Condorcet winner (0-based):", cw)  # None if not found

Has Condorcet winner: True
Condorcet winner (0-based): 0


## 7 — Ordinal preference profiles and voting rules

In [ ]:
# Define three alternatives: status quo and two candidates
candidate_a = np.array([60.0, 80.0])
candidate_b = np.array([110.0, 50.0])
alts = np.array([sq[0], sq[1], candidate_a[0], candidate_a[1], candidate_b[0], candidate_b[1]])

prof = scl.profile_build_spatial(alts, voters)
print("Dims:", prof.dims())       # (n_voters, n_alts)
print("Rankings:\n", prof.export())

Dims: (5, 3)
Rankings:
 [[2 0 1]
 [0 2 1]
 [1 0 2]
 [2 0 1]
 [0 1 2]]


In [ ]:
print("Plurality scores:", scl.plurality_scores(prof))
print("Plurality winner:", scl.plurality_one_winner(prof))

Plurality scores: [2 1 2]
Plurality winner: 0


In [ ]:
print("Borda scores:  ", scl.borda_scores(prof))
print("Borda ranking: ", scl.borda_ranking(prof))  # best first

Borda scores:   [7 3 5]
Borda ranking:  [0 2 1]


In [ ]:
# Approval voting: each voter approves their top 2
print("Top-2 approval scores:", scl.approval_scores_topk(prof, k=2))
print("Top-2 approval winners:", scl.approval_all_winners_topk(prof, k=2))

Top-2 approval scores: [5 2 3]
Top-2 approval winners: [0]


## 8 — Pareto efficiency and Condorcet consistency

In [ ]:
print("Pareto set:", scl.pareto_set(prof))           # 0-based indices
print("Has Condorcet winner (profile):", scl.has_condorcet_winner_profile(prof))
print("Condorcet winner (profile):",     scl.condorcet_winner_profile(prof))

for i in range(prof.dims()[1]):
    print(f"  alt {i}: Condorcet-consistent = {scl.is_condorcet_consistent(prof, i)},"
          f" majority-consistent = {scl.is_majority_consistent(prof, i)}")

Pareto set: [0 1 2]
Has Condorcet winner (profile): True
Condorcet winner (profile): 0
  alt 0: Condorcet-consistent = True, majority-consistent = True
  alt 1: Condorcet-consistent = False, majority-consistent = False
  alt 2: Condorcet-consistent = False, majority-consistent = False


## 9 — Interactive visualization

`socialchoicelab.plots` provides a thin Plotly-based visualization layer.
All functions return `plotly.graph_objects.Figure` objects that render inline
in Jupyter, can be exported to self-contained HTML, or composed by feeding
the figure from one function into the next.

```python
import socialchoicelab.plots as sclp
```

The layer functions all follow the same pattern:

```python
fig = sclp.plot_spatial_voting(voters, alts, sq=sq)  # base plot
fig = sclp.layer_winset(fig, ws)                     # add layer
fig = sclp.layer_uncovered_set(fig, bnd)             # add another layer
fig.show()
```

In [ ]:
import socialchoicelab.plots as sclp

# 9.1 — Base plot: voters and status quo
fig = sclp.plot_spatial_voting(
    voters,
    sq          = sq,
    voter_names = [f"V{i}" for i in range(5)],
    title       = "5-voter legislature",
)
fig.show()

In [ ]:
# 9.2 — Uncovered set boundary
bnd = scl.uncovered_set_boundary_2d(voters, grid_resolution=12)
print(f"Uncovered set boundary: {len(bnd)} vertices")

fig = sclp.plot_spatial_voting(voters, sq=sq, title="Uncovered set (continuous)")
fig = sclp.layer_uncovered_set(fig, bnd)
fig.show()

Uncovered set boundary: 1 vertices


In [ ]:
# 9.3 — Convex hull of voter ideal points (= Pareto set under Euclidean prefs)
hull = scl.convex_hull_2d(voters)
print(f"Convex hull: {len(hull)} vertices")

fig = sclp.plot_spatial_voting(voters, sq=sq, title="Convex hull of voter ideal points")
fig = sclp.layer_convex_hull(fig, hull)
fig.show()

Convex hull: 4 vertices


In [ ]:
# 9.4 — Yolk (manually specified center and radius)
# The yolk is the smallest circle that intersects all median hyperplanes.
# Supply center and radius directly (e.g., from a numerical estimate or paper).
fig = sclp.plot_spatial_voting(voters, sq=sq, title="Yolk (approximate)")
fig = sclp.layer_yolk(fig, center_x=70.0, center_y=58.0, radius=20.0)
fig.show()

In [ ]:
# 9.5 — Full composite plot: all layers at once
fig = sclp.plot_spatial_voting(
    voters,
    sq          = sq,
    voter_names = [f"V{i}" for i in range(5)],
    title       = "Full spatial analysis",
)
fig = sclp.layer_winset(fig, ws)
fig = sclp.layer_uncovered_set(fig, bnd)
fig = sclp.layer_convex_hull(fig, hull)
fig = sclp.layer_yolk(fig, center_x=70.0, center_y=58.0, radius=20.0)
fig.show()

---
*End of notebook.  See `python/README.md` for local setup instructions.*